In [6]:
# !pip install -q transformers datasets peft accelerate pandas

In [7]:
import pandas as pd
import json
import re
import itertools
import os
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

In [8]:
INPUT_FILENAME = "ddq_document_v2.xlsx" 
OUTPUT_FILENAME = "train_data.jsonl"

def clean_text(text):
    if not isinstance(text, str): return ""
    return text.strip().replace('\n', ' ')

def prepare_training_data(INPUT_FILENAME=INPUT_FILENAME, output_filename=OUTPUT_FILENAME):
    
    INPUT_FILE = INPUT_FILENAME
    
    print(f"[*] Đang đọc file dữ liệu: {INPUT_FILE}")
    
    if INPUT_FILE.endswith('.csv'):
        df = pd.read_csv(INPUT_FILE)
    else:
        df = pd.read_excel(INPUT_FILE)

    training_pairs = []
    
    for _, row in df.iterrows():
        examples_raw = str(row.get('examples', ''))
        
        if examples_raw == 'nan': continue
            
        example_list = re.split(r'[\n]+', examples_raw)
        
        clean_examples = [ex.strip() for ex in example_list if len(ex.strip()) > 5]
        
        if len(clean_examples) < 2:
            continue
            
        pairs = list(itertools.permutations(clean_examples, 2))
        
        for inp, out in pairs:
            training_pairs.append({
                "input": f"paraphrase: {inp}", 
                "target": out
            })
    
    print(f"[*] Đang lưu {len(training_pairs)} cặp dữ liệu Paraphrase...")
    with open(output_filename, 'w', encoding='utf-8') as f:
        for entry in training_pairs:
            json.dump(entry, f, ensure_ascii=False)
            f.write('\n')
            
    print(f"Đã tạo {output_filename}. Sẵn sàng train lại!")
current_dir = os.getcwd()
file_path = os.path.join(current_dir, OUTPUT_FILENAME)

if os.path.exists(file_path):
    prepare_training_data()
else:
    xlsx_files = [f for f in os.listdir(current_dir) if f.endswith('.xlsx')]
    if xlsx_files:
        print(f"⚠ Không thấy {INPUT_FILENAME}, đang dùng file: {xlsx_files[0]}")
        prepare_training_data()
    else:
        print(f"X LỖI: Không tìm thấy file Excel nào")

⚠ Không thấy ddq_document_v2.xlsx, đang dùng file: ddq_document_v2.xlsx
[*] Đang đọc file dữ liệu: ddq_document_v2.xlsx
[*] Đang lưu 29548 cặp dữ liệu Paraphrase...
Đã tạo train_data.jsonl. Sẵn sàng train lại!


In [9]:
# Load Dataset & Tokenizer
dataset = load_dataset("json", data_files="train_data.jsonl")

full_dataset = dataset['train'].train_test_split(test_size=0.1)

print("Mẫu dữ liệu:", full_dataset['train'][0])

model_id = "chieunq/vietnamese-sentence-paraphase"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

def preprocess_function(examples):
    inputs = examples["input"]
    targets = examples["target"]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = full_dataset.map(preprocess_function, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Mẫu dữ liệu: {'input': 'paraphrase: Thông tin khách hàng của tôi được khởi tạo vào ngày nào?', 'target': 'Kiểm tra nơi quản lý hồ sơ khách hàng của tôi'}


Map:   0%|          | 0/26593 [00:00<?, ? examples/s]

Map:   0%|          | 0/2955 [00:00<?, ? examples/s]

In [10]:
# LoRA config & train
lora_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = Seq2SeqTrainingArguments(
    output_dir="./fine_tuned_banking_ai",
    learning_rate=1e-3,
    per_device_train_batch_size=8, 
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    fp16=True, 
    use_cpu=False,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Bắt đầu training...")
trainer.train()

trainable params: 1,769,472 || all params: 255,442,176 || trainable%: 0.6927


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21972\3868293769.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Bắt đầu training...


Epoch,Training Loss,Validation Loss
1,0.099100,0.073314
2,0.065200,0.045722
3,0.050900,0.034343


TrainOutput(global_step=9975, training_loss=0.08687198151323132, metrics={'train_runtime': 16702.6445, 'train_samples_per_second': 4.776, 'train_steps_per_second': 0.597, 'total_flos': 1.395244673335296e+16, 'train_loss': 0.08687198151323132, 'epoch': 3.0})

In [11]:
def generate_answer(text):
    input_text = f"paraphrase: {text}"
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(input_ids=inputs["input_ids"], max_length=128, num_beams=5)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("-" * 50)
print("Input: Tra cứu các giao dịch chuyển khoản trong tuần vừa rồi")
print("Output:", generate_answer("Tra cứu các giao dịch chuyển khoản trong tuần vừa rồi"))
print("-" * 50)

model.save_pretrained("my_banking_lora_adapter")
tokenizer.save_pretrained("my_banking_lora_adapter")
# !zip -r my_banking_ai.zip my_banking_lora_adapter

--------------------------------------------------
Input: Tra cứu các giao dịch chuyển khoản trong tuần vừa rồi
Output: Tra cứu lịch sử chuyển tiền trong kỳ sao kê gần nhất,
--------------------------------------------------


('my_banking_lora_adapter\\tokenizer_config.json',
 'my_banking_lora_adapter\\special_tokens_map.json',
 'my_banking_lora_adapter\\spiece.model',
 'my_banking_lora_adapter\\added_tokens.json',
 'my_banking_lora_adapter\\tokenizer.json')